In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/model-212.keras
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/model-211.keras
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/config.json
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/X_train_pre.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/camembert_training_history.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/images_analysis.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/model-213.keras
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/history_212.pkl
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/lang.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/history_211.pkl
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/translation.json
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/df_train_clean.csv
/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/label_mapping.json
/kaggle/input/datasets/tarikouhsine/

KeyboardInterrupt: 

# CELLULE 1 — Imports & Chemins

In [5]:
import os, pickle, shutil, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ── Chemins Kaggle ───────────────────────────────────────────────────────────
INPUT_DIR  = Path('/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed')
IMAGE_DIR  = INPUT_DIR / 'images' / 'images' / 'image_train'
WORK_DIR   = Path('/kaggle/working')
OUTPUT_DIR = WORK_DIR / 'streamlit-data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random_state = 42
print(f"✅ Chemins configurés")
print(f"   Images  : {IMAGE_DIR} ({len(list(IMAGE_DIR.glob('*.jpg'))):,} fichiers)")
print(f"   Sortie  : {OUTPUT_DIR}")

✅ Chemins configurés
   Images  : /kaggle/input/datasets/tarikouhsine/rakuten-preprocessed/images/images/image_train (84,916 fichiers)
   Sortie  : /kaggle/working/streamlit-data


# CELLULE 2 — Fonctions utilitaires

In [7]:
def get_average_pred(img_pred, text_pred,
                     img_pred_weight=0.5, text_pred_weight=0.5):
    """Fusion pondérée des probabilités image + texte."""
    combined_pred = (img_pred * img_pred_weight) + (text_pred * text_pred_weight)
    return np.argmax(combined_pred, axis=1)

CODE2LABEL = {
    10:'Livre occasion', 40:'Jeu vidéo, accessoire tech',
    50:'Accessoire Console', 60:'Console de jeu',
    1140:'Figurine', 1160:'Carte Collection',
    1180:'Jeu Plateau', 1280:'Jouet enfant, déguisement',
    1281:'Jeu société', 1300:'Jouet tech',
    1301:'Jeu intérieur, chaussettes', 1302:'Jeu extérieur, vêtement',
    1320:'Bébé', 1560:'Mobilier intérieur',
    1920:'Chambre', 1940:'Cuisine',
    2060:'Décoration intérieure', 2220:'Animal',
    2280:'Revues, journaux', 2403:'Magazines, livres, BDs',
    2462:'Jeu occasion', 2522:'Bureautique, papeterie',
    2582:'Mobilier extérieur', 2583:'Piscine',
    2585:'Bricolage', 2705:'Livre neuf', 2905:'Jeu PC',
}
print("✅ Fonctions prêtes")

✅ Fonctions prêtes


# CELLULE 3 — Chargement des données et des probabilités

In [8]:
df_test = pd.read_csv(INPUT_DIR / 'df_test.csv', index_col=0)

# Construction colonne imagefile si absente
if 'imagefile' not in df_test.columns:
    df_test['imagefile'] = ('image_' + df_test['imageid'].astype(str) +
                            '_product_' + df_test['productid'].astype(str) + '.jpg')

# Classes ordonnées (entiers, comme dans le notebook fusion)
classes_ordered = sorted(df_test['prdtypecode'].unique().tolist())
print(f"Dataset test     : {df_test.shape}")
print(f"Classes          : {len(classes_ordered)}")
display(df_test.head(3))

# ── Probabilités sauvegardées par le notebook fusion ─────────────────────────
# Ces fichiers sont produits par le notebook 7.0-modeling-fusion.ipynb
# et doivent être ajoutés comme dataset Kaggle (ou se trouver dans /kaggle/working)

# Cherche d'abord dans le dataset d'entrée, sinon dans working
def load_probs(filename):
    for base in [INPUT_DIR, WORK_DIR]:
        p = base / filename
        if p.exists():
            arr = np.load(p)
            print(f"   ✅ {filename} chargé depuis {base} — shape : {arr.shape}")
            return arr
    print(f"   ⚠️  {filename} non trouvé")
    return None

print("\nChargement des probabilités :")
test_pred_prob_image = load_probs('vgg16_test_probs.npy')
test_pred_prob_text  = load_probs('camembert_test_probs.npy')

Dataset test     : (8492, 8)
Classes          : 27


,productid,imageid,text_fr,lang,categorie,imagefile,text_lemmatise,prdtypecode
29927,3469563513,1215209457,Adventure Parks Heavy Duty Tree Strap - Dont t...,en,"Jouet enfant, déguisement",image_1215209457_product_3469563513.jpg,adventure park heavy duty tree strap dont tie ...,1280
42596,4152889175,1309607798,Magideal 2 Pcs Orff Instrument De Musique Hoch...,fr,Autour du bébé,image_1309607798_product_4152889175.jpg,magideal pc orff instrument musique hochets ma...,1320
158,1190507078,1068211845,Casa Padrino Baroque Dining Room avec des acco...,en,Mobilier intérieur,image_1068211845_product_1190507078.jpg,casa padrino baroque din room accoudoirs motif...,1560



Chargement des probabilités :
   ✅ vgg16_test_probs.npy chargé depuis /kaggle/input/datasets/tarikouhsine/rakuten-preprocessed — shape : (8492, 27)
   ✅ camembert_test_probs.npy chargé depuis /kaggle/input/datasets/tarikouhsine/rakuten-preprocessed — shape : (8492, 27)


# CELLULE 4 — Calcul des prédictions et colonnes de correction

In [9]:
# Prédictions individuelles (indices → classes)
test_pred_image_idx = np.argmax(test_pred_prob_image, axis=1)
test_pred_text_idx  = np.argmax(test_pred_prob_text,  axis=1)

# Fusion pondérée — mêmes poids que dans le notebook fusion
# Modifie img_pred_weight selon tes meilleurs poids trouvés en grid search
IMG_WEIGHT  = 0.15   # ajuste selon résultats du notebook fusion
TEXT_WEIGHT = 0.85

test_pred_fusion_idx = get_average_pred(
    test_pred_prob_image, test_pred_prob_text,
    img_pred_weight=IMG_WEIGHT,
    text_pred_weight=TEXT_WEIGHT
)

# Conversion indices → codes produit
test_pred_image  = [classes_ordered[c] for c in test_pred_image_idx]
test_pred_text   = [classes_ordered[c] for c in test_pred_text_idx]
test_pred_fusion = [classes_ordered[c] for c in test_pred_fusion_idx]

# Colonnes de correction
df_test['prediction_correct_image']  = df_test['prdtypecode'] == test_pred_image
df_test['prediction_correct_text']   = df_test['prdtypecode'] == test_pred_text
df_test['prediction_correct_fusion'] = df_test['prdtypecode'] == test_pred_fusion

# Statistiques
print("=" * 50)
print("ACCURACY PAR MODALITÉ")
print("=" * 50)
print(f"   Image seul  (VGG16)     : {df_test['prediction_correct_image'].mean():.4f}")
print(f"   Texte seul  (CamemBERT) : {df_test['prediction_correct_text'].mean():.4f}")
print(f"   Fusion ({IMG_WEIGHT}/{TEXT_WEIGHT})          : {df_test['prediction_correct_fusion'].mean():.4f}")

ACCURACY PAR MODALITÉ
   Image seul  (VGG16)     : 0.6538
   Texte seul  (CamemBERT) : 0.8781
   Fusion (0.15/0.85)          : 0.8813


# CELLULE 5 — Sauvegarde du dataframe complet

In [10]:
df_test.to_csv(OUTPUT_DIR / 'df_streamlit.csv')
print(f"✅ df_streamlit.csv sauvegardé → {OUTPUT_DIR / 'df_streamlit.csv'}")

df_streamlit = pd.read_csv(OUTPUT_DIR / 'df_streamlit.csv', index_col=0)
display(df_streamlit.head(3))

✅ df_streamlit.csv sauvegardé → /kaggle/working/streamlit-data/df_streamlit.csv


,productid,imageid,text_fr,lang,categorie,imagefile,text_lemmatise,prdtypecode,prediction_correct_image,prediction_correct_text,prediction_correct_fusion
29927,3469563513,1215209457,Adventure Parks Heavy Duty Tree Strap - Dont t...,en,"Jouet enfant, déguisement",image_1215209457_product_3469563513.jpg,adventure park heavy duty tree strap dont tie ...,1280,False,False,False
42596,4152889175,1309607798,Magideal 2 Pcs Orff Instrument De Musique Hoch...,fr,Autour du bébé,image_1309607798_product_4152889175.jpg,magideal pc orff instrument musique hochets ma...,1320,False,True,True
158,1190507078,1068211845,Casa Padrino Baroque Dining Room avec des acco...,en,Mobilier intérieur,image_1068211845_product_1190507078.jpg,casa padrino baroque din room accoudoirs motif...,1560,True,True,True


# CELLULE 6 — Sélection du subset représentatif

In [11]:
# 5 exemples de chaque combinaison image/texte/fusion
# pour illustrer l'apport de chaque modalité dans la démo Streamlit

df = df_streamlit

df_streamlit_subset = pd.concat([
    # Fusion gagne sans image ni texte (rare — synergies de probabilités)
    df[( df['prediction_correct_fusion']
       & ~df['prediction_correct_image']
       & ~df['prediction_correct_text'])].head(5),

    # Fusion gagne grâce à l'image seule
    df[( df['prediction_correct_fusion']
       &  df['prediction_correct_image']
       & ~df['prediction_correct_text'])].head(5),

    # Fusion gagne grâce au texte seul
    df[( df['prediction_correct_fusion']
       & ~df['prediction_correct_image']
       &  df['prediction_correct_text'])].head(5),

    # Les trois corrects ensemble
    df[( df['prediction_correct_fusion']
       &  df['prediction_correct_image']
       &  df['prediction_correct_text'])].head(5),
], axis=0)

print(f"Subset Streamlit : {len(df_streamlit_subset)} produits")
print(f"\nRépartition :")
print(f"   Fusion ✅ / Image ❌ / Texte ❌ : "
      f"{(~df_streamlit_subset['prediction_correct_image'] & ~df_streamlit_subset['prediction_correct_text']).sum()}")
print(f"   Fusion ✅ / Image ✅ / Texte ❌ : "
      f"{( df_streamlit_subset['prediction_correct_image'] & ~df_streamlit_subset['prediction_correct_text']).sum()}")
print(f"   Fusion ✅ / Image ❌ / Texte ✅ : "
      f"{(~df_streamlit_subset['prediction_correct_image'] &  df_streamlit_subset['prediction_correct_text']).sum()}")
print(f"   Fusion ✅ / Image ✅ / Texte ✅ : "
      f"{( df_streamlit_subset['prediction_correct_image'] &  df_streamlit_subset['prediction_correct_text']).sum()}")

display(df_streamlit_subset)

Subset Streamlit : 17 produits

Répartition :
   Fusion ✅ / Image ❌ / Texte ❌ : 2
   Fusion ✅ / Image ✅ / Texte ❌ : 5
   Fusion ✅ / Image ❌ / Texte ✅ : 5
   Fusion ✅ / Image ✅ / Texte ✅ : 5


,productid,imageid,text_fr,lang,categorie,imagefile,text_lemmatise,prdtypecode,prediction_correct_image,prediction_correct_text,prediction_correct_fusion
710,3027358213,1307802838,Papelera mini 175 mm Spiderman - Papelera mini...,id,Autour du bébé,image_1307802838_product_3027358213.jpg,papelera mini mm spiderman papelera mini mm sp...,1320,False,False,True
556,160284580,916824168,Il suffit de danser les plus grands succès -,en,"Jeu vidéo, accessoire tech.",image_916824168_product_160284580.jpg,suffit danser plus grand succès,40,False,False,True
30775,3439675612,1213061980,Statuette The Witcher 3: The Wild Hunt - Shani...,fr,Figurine,image_1213061980_product_3439675612.jpg,statuette witcher wild hunt shani statuette wi...,1140,True,False,True
70100,3893160908,1260254826,Mini 45 Pouces Arcade Jeu Retro Machines Pour ...,fr,Accessoire Console,image_1260254826_product_3893160908.jpg,mini pouces arcade jeu retro machine enfants c...,50,True,False,True
26427,985690380,1052833103,Chien Sac À Pijama -,hr,"Jouet enfant, déguisement",image_1052833103_product_985690380.jpg,chien sac pijama,1280,True,False,True
31444,1913791872,1277819334,Composition Florale Roses Artificielles Blanc ...,fr,Décoration intérieure,image_1277819334_product_1913791872.jpg,composition florale rose artificielles blanc r...,2060,True,False,True
37114,51507312,493909730,Tout L'univers N° 102 Du 2 Au 9 Octobre 1963 -,fr,Revues et journaux,image_493909730_product_51507312.jpg,tout univers octobre,2280,True,False,True
42596,4152889175,1309607798,Magideal 2 Pcs Orff Instrument De Musique Hoch...,fr,Autour du bébé,image_1309607798_product_4152889175.jpg,magideal pc orff instrument musique hochets ma...,1320,False,True,True
39959,2009635938,1120113941,Vb601 2.4g Bébé Moniteur Vidéosurveillance Vid...,fr,Autour du bébé,image_1120113941_product_2009635938.jpg,vb601 4g bébé moniteur vidéosurveillance vidéo...,1320,False,True,True
3901,255464705,968890453,Tableau Lumineux Jeune Fille En Forme De Fée -,fr,Décoration intérieure,image_968890453_product_255464705.jpg,tableau lumineux jeune fille forme fée,2060,False,True,True


# CELLULE 7 — Copie des images du subset

In [13]:
# Copie les images du subset dans un dossier temporaire
# organisé par code produit (pour compatibilité Streamlit)
TMP_IMG_DIR = Path('/tmp/images_streamlit')
TMP_IMG_DIR.mkdir(parents=True, exist_ok=True)

missing = 0
for idx, row in tqdm(df_streamlit_subset.iterrows(), total=len(df_streamlit_subset)):
    # Image source : répertoire plat dans notre pipeline
    src = IMAGE_DIR / row['imagefile']

    # Destination : organisée par code produit (comme dans le projet original)
    dest_folder = TMP_IMG_DIR / str(row['prdtypecode'])
    dest_folder.mkdir(parents=True, exist_ok=True)
    dest = dest_folder / row['imagefile']

    if src.exists():
        shutil.copyfile(src, dest)
    else:
        print(f"⚠️  Image manquante : {src}")
        missing += 1

print(f"\n✅ {len(df_streamlit_subset) - missing} images copiées")
if missing:
    print(f"   ⚠️ {missing} images manquantes")

100%|██████████| 17/17 [00:00<00:00, 397.67it/s]


✅ 17 images copiées


# CELLULE 8 — Création du zip et sauvegarde finale

In [14]:
def zipdir(path, ziph):
    for root, dirs, files in os.walk(path):
        print(f"   Compression de {Path(root).name}...")
        for file in tqdm(files):
            full_path = os.path.join(root, file)
            arcname   = os.path.relpath(full_path, os.path.join(str(path), '..'))
            ziph.write(full_path, arcname)

ZIP_PATH = Path('/tmp/images_streamlit.zip')

print("⏳ Création du zip des images...")
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipdir(TMP_IMG_DIR, zipf)

# Copie dans output
shutil.copyfile(ZIP_PATH, OUTPUT_DIR / 'images_streamlit.zip')
df_streamlit_subset.to_csv(OUTPUT_DIR / 'df_streamlit_subset.csv')

print(f"\n✅ Fichiers sauvegardés dans {OUTPUT_DIR} :")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"   {f.name:<35} {f.stat().st_size / 1024:.1f} KB")

⏳ Création du zip des images...
   Compression de images_streamlit...


0it [00:00, ?it/s]


   Compression de 40...


100%|██████████| 1/1 [00:00<00:00, 166.10it/s]


   Compression de 1320...


...

   Compression de 1302...


100%|██████████| 2/2 [00:00<00:00, 560.25it/s]


   Compression de 2060...


100%|██████████| 2/2 [00:00<00:00, 699.23it/s]


✅ Fichiers sauvegardés dans /kaggle/working/streamlit-data :
   df_streamlit.csv                    9507.1 KB
   df_streamlit_subset.csv             14.3 KB
   images_streamlit.zip                423.4 KB


# CELLULE 9 — Vérification finale

In [15]:
df_check = pd.read_csv(OUTPUT_DIR / 'df_streamlit_subset.csv', index_col=0)

print(f"✅ Vérification du subset Streamlit")
print(f"   Lignes       : {len(df_check)}")
print(f"   Colonnes     : {df_check.columns.tolist()}")
print(f"\nFichiers produits pour Streamlit :")
print(f"   df_streamlit.csv        → dataset complet avec colonnes de correction")
print(f"   df_streamlit_subset.csv → 20 produits représentatifs pour la démo")
print(f"   images_streamlit.zip    → images correspondantes organisées par catégorie")

display(df_check)

✅ Vérification du subset Streamlit
   Lignes       : 17
   Colonnes     : ['productid', 'imageid', 'text_fr', 'lang', 'categorie', 'imagefile', 'text_lemmatise', 'prdtypecode', 'prediction_correct_image', 'prediction_correct_text', 'prediction_correct_fusion']

Fichiers produits pour Streamlit :
   df_streamlit.csv        → dataset complet avec colonnes de correction
   df_streamlit_subset.csv → 20 produits représentatifs pour la démo
   images_streamlit.zip    → images correspondantes organisées par catégorie


,productid,imageid,text_fr,lang,categorie,imagefile,text_lemmatise,prdtypecode,prediction_correct_image,prediction_correct_text,prediction_correct_fusion
710,3027358213,1307802838,Papelera mini 175 mm Spiderman - Papelera mini...,id,Autour du bébé,image_1307802838_product_3027358213.jpg,papelera mini mm spiderman papelera mini mm sp...,1320,False,False,True
556,160284580,916824168,Il suffit de danser les plus grands succès -,en,"Jeu vidéo, accessoire tech.",image_916824168_product_160284580.jpg,suffit danser plus grand succès,40,False,False,True
30775,3439675612,1213061980,Statuette The Witcher 3: The Wild Hunt - Shani...,fr,Figurine,image_1213061980_product_3439675612.jpg,statuette witcher wild hunt shani statuette wi...,1140,True,False,True
70100,3893160908,1260254826,Mini 45 Pouces Arcade Jeu Retro Machines Pour ...,fr,Accessoire Console,image_1260254826_product_3893160908.jpg,mini pouces arcade jeu retro machine enfants c...,50,True,False,True
26427,985690380,1052833103,Chien Sac À Pijama -,hr,"Jouet enfant, déguisement",image_1052833103_product_985690380.jpg,chien sac pijama,1280,True,False,True
31444,1913791872,1277819334,Composition Florale Roses Artificielles Blanc ...,fr,Décoration intérieure,image_1277819334_product_1913791872.jpg,composition florale rose artificielles blanc r...,2060,True,False,True
37114,51507312,493909730,Tout L'univers N° 102 Du 2 Au 9 Octobre 1963 -,fr,Revues et journaux,image_493909730_product_51507312.jpg,tout univers octobre,2280,True,False,True
42596,4152889175,1309607798,Magideal 2 Pcs Orff Instrument De Musique Hoch...,fr,Autour du bébé,image_1309607798_product_4152889175.jpg,magideal pc orff instrument musique hochets ma...,1320,False,True,True
39959,2009635938,1120113941,Vb601 2.4g Bébé Moniteur Vidéosurveillance Vid...,fr,Autour du bébé,image_1120113941_product_2009635938.jpg,vb601 4g bébé moniteur vidéosurveillance vidéo...,1320,False,True,True
3901,255464705,968890453,Tableau Lumineux Jeune Fille En Forme De Fée -,fr,Décoration intérieure,image_968890453_product_255464705.jpg,tableau lumineux jeune fille forme fée,2060,False,True,True
